# 01 · 数据审计 (Himawari-8 V7 dataset)

本 notebook 完成 P0 阶段的核心交付物：

1. **样本量地图**：按 (年, 月) 统计样本数 / 加载失败数；
2. **通道分布画像**：B08–B13 在全数据集上的分位数表与直方图；
3. **对流强度分桶**：用 B13 最低值 (norm 域) 把样本分为 deep / strong / active / weak；
4. **坏帧检出 → 黑名单扩充**：自动产生新版 `problematic_checkpoints.csv`；
5. **采样器健康检查**：跑一次加权采样器，观察实际采样分布是否符合预期。

> 运行前请确保已经 `pip install -e ".[dev]"` 且 `XN_TRAIN_DIR/XN_VAL_DIR/XN_TEST_DIR` 指向正确的数据目录。

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 让 notebook 直接 import 仓库代码（即使没有 pip install -e）
REPO_ROOT = Path.cwd().resolve().parents[0]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.h8_dataset import H8Dataset, load_blacklist
from src.data.normalizers import BAND_ORDER, NORM_LIMITS, b13_norm_threshold_for_kelvin
from src.data.samplers import IntensityBin, SamplerConfig, StratifiedConvectiveSampler, scan_dataset
from src.data.statistics import (
    QualityRule,
    audit_dataset,
    channel_distribution_summary,
    detect_problematic,
    merge_blacklist,
    monthly_summary,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)
print('Repo root :', REPO_ROOT)
print('B13 阈值映射: 240K =>', b13_norm_threshold_for_kelvin(240),
      ' 220K =>', b13_norm_threshold_for_kelvin(220),
      ' 200K =>', b13_norm_threshold_for_kelvin(200))

## 1. 数据目录与加载

In [ ]:
TRAIN_DIR = os.environ.get('XN_TRAIN_DIR', '/share/home/sera_hujun/train_data_v7_unbiased_501')
VAL_DIR   = os.environ.get('XN_VAL_DIR',   '/share/home/sera_hujun/val_data_v7_unbiased_501')
TEST_DIR  = os.environ.get('XN_TEST_DIR',  '/share/home/sera_hujun/test_data_v7_unbiased_501')
BLACKLIST_PATH = os.environ.get('XN_BLACKLIST', str(REPO_ROOT / 'problematic_checkpoints.csv'))

for tag, p in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    print(f'{tag:5s}: {p}  exists={Path(p).exists()}')
print('blacklist:', BLACKLIST_PATH, 'exists=', Path(BLACKLIST_PATH).exists())

In [ ]:
blacklist = load_blacklist(BLACKLIST_PATH)
print(f'已有黑名单条目: {len(blacklist)}')

# 不应用裁剪 / 增强：审计场景需要原始 501x501
ds_train = H8Dataset(TRAIN_DIR, mode='raw', blacklist=blacklist)
ds_val   = H8Dataset(VAL_DIR,   mode='raw', blacklist=blacklist)
ds_test  = H8Dataset(TEST_DIR,  mode='raw', blacklist=blacklist)

print(f'train: {len(ds_train):>6} 样本（已剔除 {ds_train.skipped_blacklist} 黑名单）')
print(f'val  : {len(ds_val):>6} 样本（已剔除 {ds_val.skipped_blacklist} 黑名单）')
print(f'test : {len(ds_test):>6} 样本（已剔除 {ds_test.skipped_blacklist} 黑名单）')

## 2. 月度样本量分布（按 split）

In [ ]:
splits = {'train': ds_train, 'val': ds_val, 'test': ds_test}
meta_dfs = {k: ds.metas_dataframe().assign(split=k) for k, ds in splits.items()}
all_meta = pd.concat(list(meta_dfs.values()), ignore_index=True)
all_meta['ym'] = all_meta['year'].astype(str) + '-' + all_meta['month'].astype(str).str.zfill(2)

month_table = (all_meta.groupby(['split', 'ym']).size().rename('n_samples')
               .reset_index().pivot(index='ym', columns='split', values='n_samples')
               .fillna(0).astype(int))
month_table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
month_table.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('月度样本量（按 split）')
ax.set_xlabel('YYYY-MM')
ax.set_ylabel('# sequences')
plt.xticks(rotation=80)
plt.tight_layout()
plt.show()

## 3. 全数据集统计（**重头**：可能耗时，建议开多进程）

In [ ]:
# 强烈建议在生产环境用 n_jobs=os.cpu_count()//2，本机若内存吃紧设为 0 顺序执行
N_JOBS = int(os.environ.get('XN_AUDIT_NJOBS', '0'))

audit_train = audit_dataset(ds_train.metas, n_jobs=N_JOBS)
audit_val   = audit_dataset(ds_val.metas,   n_jobs=N_JOBS)
audit_test  = audit_dataset(ds_test.metas,  n_jobs=N_JOBS)

audit_all = pd.concat([
    audit_train.assign(split='train'),
    audit_val.assign(split='val'),
    audit_test.assign(split='test'),
], ignore_index=True)
print('audit shape:', audit_all.shape)
audit_all.head(3)

In [ ]:
# 通道分布概览（开尔文域）
channel_distribution_summary(audit_all[audit_all['load_ok']])

In [ ]:
# 月度统计摘要（仅 train）
monthly_summary(audit_train[audit_train['load_ok']])

In [ ]:
# 各通道分布直方图
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6), sharey=True)
for ax, band in zip(axes, BAND_ORDER):
    col = f'{band}_min_K'
    ok = audit_all[audit_all['load_ok']][col]
    ax.hist(ok, bins=80, alpha=0.85)
    ax.axvline(NORM_LIMITS[band][0], color='r', ls='--', lw=1, label='phys_min')
    ax.axvline(NORM_LIMITS[band][1], color='g', ls='--', lw=1, label='phys_max')
    ax.set_title(f'{band} : per-sequence MIN (K)')
    ax.set_xlabel('K')
axes[0].set_ylabel('count')
axes[-1].legend()
plt.tight_layout()
plt.show()

## 4. 对流强度分桶 (基于 B13 norm-min)

In [ ]:
BINS = (
    IntensityBin('deep', b13_norm_threshold_for_kelvin(200)),
    IntensityBin('strong', b13_norm_threshold_for_kelvin(220)),
    IntensityBin('active', b13_norm_threshold_for_kelvin(240)),
    IntensityBin('weak', 1.0),
)
scan_train = scan_dataset(ds_train.metas, BINS, n_jobs=N_JOBS)

bucket_dist = (scan_train.groupby(['month', 'intensity_bin']).size()
                .rename('n').reset_index()
                .pivot(index='month', columns='intensity_bin', values='n')
                .reindex(columns=['deep', 'strong', 'active', 'weak']).fillna(0).astype(int))
bucket_dist

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bucket_dist.plot(kind='bar', stacked=True, ax=ax,
                 color=['#9b1c1c', '#e98a3a', '#f0c419', '#cdcdcd'])
ax.set_title('Train: 对流强度分桶（按月份）')
ax.set_ylabel('# sequences')
plt.tight_layout()
plt.show()

## 5. 坏帧检出与黑名单扩充

In [ ]:
rule = QualityRule(
    valid_ratio_min=0.95,
    tearing_b13_median_K_min=295.0,
    tearing_b13_q05_K_max=200.0,
    flat_clear_b13_std_K_max=0.5,
    flat_clear_b13_mean_K_min=290.0,
)
problems = detect_problematic(audit_all, rule)
print(f'共检出问题样本: {len(problems)} / {len(audit_all)}')
problems['reason'].value_counts().head(15)

In [ ]:
# 写出新黑名单（在仓库根目录创建一个新文件，避免覆盖旧版本）
OUT_PATH = REPO_ROOT / 'problematic_checkpoints_v2.csv'
merged = merge_blacklist(
    existing_path=BLACKLIST_PATH if Path(BLACKLIST_PATH).exists() else None,
    new_problems=problems,
    output_path=OUT_PATH,
)
print(f'写入 {OUT_PATH}, 累计黑名单条目: {len(merged)}')
merged.head()

## 6. 加权采样器健康检查

构造一次 `StratifiedConvectiveSampler`，模拟一个 epoch 的抽样，验证：
- 各强度桶的实际采样比例是否符合 `bin_weights` 配置；
- 月份分布是否被 sampler 拉平。

In [ ]:
import torch

BIN_WEIGHTS = {'deep': 5.0, 'strong': 3.0, 'active': 2.0, 'weak': 1.0}
cfg = SamplerConfig(
    enable=True,
    intensity_bins=BINS,
    bin_weights=BIN_WEIGHTS,
)
g = torch.Generator().manual_seed(2025)
sampler = StratifiedConvectiveSampler(scan_train, cfg, num_samples=len(scan_train), generator=g)

indices = list(iter(sampler))
drawn = scan_train.iloc[indices]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
drawn['intensity_bin'].value_counts().reindex(['deep', 'strong', 'active', 'weak']).plot(
    kind='bar', ax=axes[0], color=['#9b1c1c', '#e98a3a', '#f0c419', '#cdcdcd'])
axes[0].set_title('采样后 强度分布')
axes[0].set_ylabel('count')
drawn.groupby('month').size().plot(kind='bar', ax=axes[1])
axes[1].set_title('采样后 月份分布')
axes[1].set_xlabel('month')
plt.tight_layout()
plt.show()

## 7. 结论与下一步

| 检查项 | 期望 | 实际 |
|---|---|---|
| 月度样本量 | 4–10 月集中、其余近 0 | 待运行确认 |
| B08–B13 分布 | 不出现持续触界（截断 < 5%） | 待运行确认 |
| 强对流桶占比 | train 集 ≥ 25% | 待运行确认 |
| 坏帧检出 | 撕裂帧 < 0.5%，加载失败 = 0 | 待运行确认 |
| 采样后强度分布 | 强对流桶被显著放大 | 待运行确认 |

确认上述各项均合理后，将新的 `problematic_checkpoints_v2.csv` 设为生产黑名单，进入：

1. PySTEPS / ConvLSTM 基线复现 (`02_baselines.ipynb`)；
2. Stage-A 时空 VAE 训练。